# Clinical Protocol Lookup Assistant
## Assignment 1B — Instruction Fine-Tuning with QLoRA [5 Marks]
### Variant 4 — Clinical Protocol Lookup Assistant (Health Domain)

**Domain:** Medical & Clinical Literature  
**Model:** BioGPT (347M) — `microsoft/biogpt`  
**Use Case:** Help clinicians, nurses, and pharmacists quickly look up treatment protocols, drug dosage guidelines, and clinical pathways.  

**Disclaimer:** Every generated response includes:  
*"This output is for educational/reference purposes only and must not replace professional clinical judgment."*

---
### Pipeline Overview (Part B)
- **Part B1** — Instruction Dataset Creation [2 Marks]
- **Part B2** — QLoRA Fine-Tuning with One Adapter Configuration [2 Marks]
- **Part B3** — Evaluation & Comparative Analysis [1 Mark]

**Prerequisite:** This notebook requires the CPT model checkpoint saved by Assignment 1A.

---
## 0. Environment Setup & Library Installation
Install all required dependencies for the pipeline.

In [18]:
# ============================================================
# Cell 0.1 — Install required packages
# ============================================================
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q trl sentencepiece protobuf sacremoses
!pip install -q PyPDF2 pymupdf langdetect datasketch
!pip install -q matplotlib pandas tqdm pyarrow
!pip install -q "torchao>=0.16.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
# ============================================================
# Cell 0.2 — Import all libraries (centralized imports)
# ============================================================
import os
import re
import json
import glob
import warnings
import time
import pathlib
from collections import Counter

# ------------------------------------------------------------------
# Windows Encoding Fix (Required for trl library on Windows)
# ------------------------------------------------------------------
if not getattr(pathlib.Path, '_utf8_patched', False):
    def _read_text_utf8(self, encoding=None, errors=None, newline=None):
        if encoding is None:
            encoding = "utf-8"
        with open(self, 'r', encoding=encoding, errors=errors, newline=newline) as f:
            return f.read()
    pathlib.Path.read_text = _read_text_utf8
    pathlib.Path._utf8_patched = True

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from datasets import load_dataset, Dataset as HFDataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings("ignore")

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cpu


In [20]:
# ============================================================
# Cell 0.3 — Configuration Constants (Single source of truth)
# ============================================================

# ----- Model Configuration -----
MODEL_ID = "microsoft/biogpt"                # BioGPT (347M params) — medical domain
MODEL_MAX_LENGTH = 1024                       # Context window for sequence packing

# ----- Directory Paths -----
PDF_DIR = "./clinical_pdfs/"                  # Raw PDF documents
CORPUS_DIR = "./domain_corpus/"               # Cleaned .txt files
OUTPUT_DIR = "./outputs/"                     # Checkpoints, datasets, results
CPT_CHECKPOINT_DIR = "./outputs/cpt_model/"   # CPT model checkpoint (from Part A)
SFT_CHECKPOINT_DIR = "./outputs/sft_model/"   # QLoRA adapter checkpoint

# ----- QLoRA Configuration -----
LORA_RANK = 16                                # Adapter B (Balanced)
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# ----- SFT Training Hyperparameters -----
SFT_EPOCHS = 3
SFT_BATCH_SIZE = 2
SFT_LEARNING_RATE = 2e-4
SFT_MAX_SEQ_LENGTH = 512

# ----- Evaluation Prompts -----
DOMAIN_PROMPTS = [
    "What is the first-line treatment for uncomplicated community-acquired pneumonia in adults?",
    "What are the recommended dosing guidelines for Vancomycin in patients with renal impairment?",
    "What are the WHO criteria for initiating antiretroviral therapy in adults with HIV?",
]

GENERAL_PROMPTS = [
    "The capital of France is",
    "Water boils at",
    "The speed of light is approximately",
]

# ----- Disclaimer -----
DISCLAIMER = "This output is for educational/reference purposes only and must not replace professional clinical judgment."

# Create directories
for d in [PDF_DIR, CORPUS_DIR, OUTPUT_DIR, CPT_CHECKPOINT_DIR, SFT_CHECKPOINT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Configuration loaded. All directories created.")

Configuration loaded. All directories created.


In [21]:
# ============================================================
# Cell 0.4 — Load Cleaned Corpus from Disk (Generated by Part A)
# ============================================================
# Part A saves cleaned .txt files in CORPUS_DIR.
# We load them here for instruction dataset generation.
# ============================================================

def load_corpus(corpus_dir: str) -> dict:
    """Load all .txt files from corpus directory."""
    corpus = {}
    for txt_file in glob.glob(os.path.join(corpus_dir, "*.txt")):
        with open(txt_file, "r", encoding="utf-8") as f:
            corpus[os.path.basename(txt_file)] = f.read()
    return corpus


corpus_clean = load_corpus(CORPUS_DIR)
print(f"Loaded {len(corpus_clean)} cleaned documents from '{CORPUS_DIR}'")
print(f"Total characters: {sum(len(t) for t in corpus_clean.values()):,}")

if len(corpus_clean) == 0:
    print("\n\u26a0\ufe0f  WARNING: No cleaned corpus found!")
    print("Please run Assignment 1A first to generate the cleaned corpus.")

Loaded 7 cleaned documents from './domain_corpus/'
Total characters: 2,622,571


In [22]:
# ============================================================
# Cell 0.5 — Load Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"[INFO] Set pad_token = eos_token ({tokenizer.eos_token})")

print(f"Tokenizer loaded: {MODEL_ID}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

Tokenizer loaded: microsoft/biogpt
Vocabulary size: 42,384


---
## Part B1 — Instruction Dataset Creation [2 Marks]

Generate instruction-response pairs from the cleaned clinical corpus.
Each response includes the mandatory clinical disclaimer.

In [23]:
# ============================================================
# Cell B1.1 — Instruction Dataset Generation Module
# ============================================================
# Method: Heuristic extraction + template-based generation
# from the cleaned clinical .txt files.
# Each entry: {"instruction": ..., "response": ...}
# All responses include the clinical disclaimer.
# ============================================================

def generate_instruction_pairs_heuristic(text: str, source_name: str) -> list:
    """
    Generate instruction-response pairs from a clinical text document
    using heuristic section-based extraction.
    """
    pairs = []
    
    # Split into paragraphs (sections)
    paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 100]
    
    # Question templates for clinical domain
    templates = [
        "What does the clinical guideline say about {topic}?",
        "Explain the recommended protocol for {topic}.",
        "What are the key considerations for {topic} according to clinical guidelines?",
        "Summarize the clinical recommendations regarding {topic}.",
        "What is the evidence-based approach for {topic}?",
        "Describe the treatment protocol for {topic}.",
        "What are the clinical guidelines for managing {topic}?",
        "What dosage and administration guidelines exist for {topic}?",
    ]
    
    for i, para in enumerate(paragraphs):
        first_sentence = para.split(".")[0].strip()
        if len(first_sentence) < 10 or len(first_sentence) > 200:
            continue
        
        topic = first_sentence.lower()
        template = templates[i % len(templates)]
        instruction = template.format(topic=topic)
        response = f"{para}\n\n{DISCLAIMER}"
        
        pairs.append({
            "instruction": instruction,
            "response": response,
        })
    
    return pairs


# --- Alternative: Synthetic generation prompt template ---
SYNTHETIC_PROMPT_TEMPLATE = """
Read the clinical text below and generate 10 instruction-response pairs in JSON format 
based ONLY on this text. Each entry must have 'instruction' and 'response' keys.
The instruction should be a clinician's question. The response should be a factual answer 
derived directly from the text. End every response with:
"This output is for educational/reference purposes only and must not replace professional clinical judgment."

Clinical Text:
{text}

Generate the JSON array:
"""

print("Instruction generation module defined.")
print(f"Synthetic prompt template available for LLM-based generation.")

Instruction generation module defined.
Synthetic prompt template available for LLM-based generation.


In [ ]:
# ============================================================
# Cell B1.2 — Generate and Save Instruction Dataset
# ============================================================

all_pairs = []

for fname, text in tqdm(corpus_clean.items(), desc="Generating instruction pairs"):
    pairs = generate_instruction_pairs_heuristic(text, fname)
    all_pairs.extend(pairs)

print(f"\nTotal instruction-response pairs generated: {len(all_pairs)}")

# Show 5 sample pairs
print(f"\n{'='*60}")
print("SAMPLE INSTRUCTION-RESPONSE PAIRS")
print(f"{'='*60}")
    print(f"\n--- Pair {i} ---")
    print(f"Instruction: {pair['instruction'][:150]}")
    print(f"Response:    {pair['response'][:200]}...")

# --- Train/Eval Split (80/20) ---
np.random.seed(42)
indices = np.random.permutation(len(all_pairs))
split_point = int(len(all_pairs) * 0.8)

train_pairs = [all_pairs[i] for i in indices[:split_point]]
eval_pairs = [all_pairs[i] for i in indices[split_point:]]

print(f"\n--- Dataset Split ---")
print(f"Training set:   {len(train_pairs)} examples")
print(f"Evaluation set: {len(eval_pairs)} examples")

# Save as JSONL
jsonl_path = os.path.join(OUTPUT_DIR, "instruction_dataset.jsonl")
with open(jsonl_path, "w", encoding="utf-8") as f:
    for pair in all_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

train_jsonl = os.path.join(OUTPUT_DIR, "instruction_train.jsonl")
eval_jsonl = os.path.join(OUTPUT_DIR, "instruction_eval.jsonl")

with open(train_jsonl, "w", encoding="utf-8") as f:
    for pair in train_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

with open(eval_jsonl, "w", encoding="utf-8") as f:
    for pair in eval_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

print(f"\nFull dataset saved to: {jsonl_path}")
print(f"Train split saved to:  {train_jsonl}")
print(f"Eval split saved to:   {eval_jsonl}")

Generating instruction pairs:   0%|          | 0/7 [00:00<?, ?it/s]


Total instruction-response pairs generated: 510

SAMPLE INSTRUCTION-RESPONSE PAIRS

--- Pair 1 ---
Instruction: What does the clinical guideline say about covid-19 rapid guideline: 
managing covid-19 
nice guideline 
published: 23 march 2021 
last updated: 1 may
Response:    COVID-19 rapid guideline: 
managing COVID-19 
NICE guideline 
Published: 23 March 2021 
Last updated: 1 May 2025 
www.nice.org.uk/guidance/ng191 
 NICE 2026. All rights reserved. Subject to Notice of ...

--- Pair 2 ---
Instruction: Explain the recommended protocol for your responsibility 
the recommendations in this guideline represent the view of nice, arrived at after careful 

Response:    Your responsibility 
The recommendations in this guideline represent the view of NICE, arrived at after careful 
consideration of the evidence available. When exercising their judgement, professionals...

--- Pair 3 ---
Instruction: What are the key considerations for contents 
overview according to clinical guidelines?
Resp

### Inference — Part B1

- **Heuristic generation** extracts paragraphs from clinical texts and pairs them with template-based questions. This ensures responses are grounded in the actual corpus.
- **Synthetic generation** (prompt template provided) can be used with an external LLM for higher-quality, more diverse question formulations.
- All responses include the mandatory disclaimer per Variant 4 requirements.
- 80/20 train/eval split with fixed seed ensures reproducibility.

---
## Part B2 — QLoRA Fine-Tuning with One Adapter Configuration [2 Marks]

Apply QLoRA (4-bit quantized base + LoRA adapters) for instruction fine-tuning.

**Adapter B (Balanced):** r=16, \u03b1=32, target: q_proj, v_proj

In [25]:
# ============================================================
# Cell B2.1 — Load Base Model for SFT (QLoRA on GPU / LoRA on CPU)
# ============================================================

import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load CPT model (or base model if CPT checkpoint unavailable)
model_path = CPT_CHECKPOINT_DIR if os.path.exists(os.path.join(CPT_CHECKPOINT_DIR, "config.json")) else MODEL_ID

if torch.cuda.is_available():
    # GPU: 4-bit quantization config for QLoRA
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    sft_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=bnb_config,
        device_map="auto",
    )
    sft_model = prepare_model_for_kbit_training(sft_model)
    print(f"Model loaded with 4-bit QLoRA quantization (GPU)")
else:
    # CPU: load in float32, no quantization
    sft_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float32,
    )
    for param in sft_model.parameters():
        param.requires_grad = False
    print(f"Model loaded in float32 (CPU mode \u2014 standard LoRA, no quantization)")

print(f"Model loaded from: {model_path}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie biogpt.embed_tokens.weight to output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded in float32 (CPU mode — standard LoRA, no quantization)
Model loaded from: ./outputs/cpt_model/


In [26]:
# ============================================================
# Cell B2.2 — Configure LoRA Adapter (Adapter B — Balanced)
# ============================================================

lora_config = LoraConfig(
    r=LORA_RANK,                         # 16 — balanced capacity
    lora_alpha=LORA_ALPHA,               # 32 — scaling factor
    lora_dropout=LORA_DROPOUT,           # 0.05 — light regularization
    target_modules=LORA_TARGET_MODULES,  # ["q_proj", "v_proj"]
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA to the model
sft_model = get_peft_model(sft_model, lora_config)

# Report adapter statistics
sft_model.print_trainable_parameters()

print(f"\n--- LoRA Adapter Configuration ---")
print(f"  Rank (r):        {LORA_RANK}")
print(f"  Alpha (\u03b1):       {LORA_ALPHA}")
print(f"  Dropout:         {LORA_DROPOUT}")
print(f"  Target modules:  {LORA_TARGET_MODULES}")
print(f"  Adapter type:    Adapter B (Balanced)")

trainable params: 1,572,864 || all params: 391,737,344 || trainable%: 0.4015

--- LoRA Adapter Configuration ---
  Rank (r):        16
  Alpha (α):       32
  Dropout:         0.05
  Target modules:  ['q_proj', 'v_proj']
  Adapter type:    Adapter B (Balanced)


In [27]:
# ============================================================
# Cell B2.3 — Prepare Training Dataset with Chat Template
# ============================================================

def format_instruction(sample: dict) -> str:
    """Format a single instruction-response pair into a training prompt."""
    return f"""### Instruction:
{sample['instruction']}

### Response:
{sample['response']}"""


# Load training and evaluation datasets
train_dataset = load_dataset("json", data_files=train_jsonl, split="train")
eval_dataset = load_dataset("json", data_files=eval_jsonl, split="train")

print(f"Training examples: {len(train_dataset)}")
print(f"Evaluation examples: {len(eval_dataset)}")
print(f"\nSample formatted prompt:")
print(format_instruction(train_dataset[0])[:300])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Training examples: 408
Evaluation examples: 102

Sample formatted prompt:
### Instruction:
What does the clinical guideline say about who consolidated guidelines on tuberculosis: 
drug-susceptible tuberculosis treatment
viii
pre-xdr-tb:3 tb caused by mycobacterium tuberculosis (m?

### Response:
WHO consolidated guidelines on tuberculosis: 
drug-susceptible tuberculosis t


In [28]:
# ============================================================
# Cell B2.4 — LoRA/QLoRA SFT Training Execution
# ============================================================

is_cpu = not torch.cuda.is_available()

# CPU adaptations
sft_max_steps = 10 if is_cpu else -1
sft_grad_accum = 1 if is_cpu else 4
sft_bs = 1 if is_cpu else SFT_BATCH_SIZE
sft_optim = "adamw_torch" if is_cpu else "paged_adamw_8bit"

sft_config = SFTConfig(
    output_dir=SFT_CHECKPOINT_DIR,
    
    # Training schedule
    num_train_epochs=SFT_EPOCHS if not is_cpu else 1,
    max_steps=sft_max_steps,
    per_device_train_batch_size=sft_bs,
    per_device_eval_batch_size=sft_bs,
    gradient_accumulation_steps=sft_grad_accum,
    
    # Optimizer
    learning_rate=SFT_LEARNING_RATE,
    warmup_steps=5 if is_cpu else 20,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim=sft_optim,
    
    # Precision
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=False,
    
    # SFT-specific
    max_length=SFT_MAX_SEQ_LENGTH,
    packing=False,
    dataset_text_field="text",
    
    # Logging & Evaluation
    logging_steps=2 if is_cpu else 10,
    eval_strategy="no" if is_cpu else "epoch",
    save_strategy="no" if is_cpu else "epoch",
    save_total_limit=2,
    report_to="none",
    
    # Misc
    remove_unused_columns=False,
    group_by_length=True,
    use_cpu=is_cpu,
)

# Initialize SFTTrainer
sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    formatting_func=format_instruction,
)

print(f"{'='*60}")
print(f"STARTING LoRA INSTRUCTION FINE-TUNING")
print(f"{'='*60}")
print(f"  Model:          {model_path}")
print(f"  Adapter:        B (Balanced) \u2014 r={LORA_RANK}, \u03b1={LORA_ALPHA}")
print(f"  Max steps:      {sft_max_steps} (-1 = all)")
print(f"  Batch size:     {sft_bs} (effective: {sft_bs * sft_grad_accum})")
print(f"  Learning rate:  {SFT_LEARNING_RATE}")
print(f"  Max seq length: {SFT_MAX_SEQ_LENGTH}")
if is_cpu:
    print(f"  \u26a0\ufe0f  CPU mode: limited to {sft_max_steps} steps")

# Train
sft_result = sft_trainer.train()

print(f"\n{'='*60}")
print(f"LoRA SFT TRAINING COMPLETE")
print(f"{'='*60}")
print(f"  Final loss: {sft_result.training_loss:.4f}")
print(f"  Total steps: {sft_trainer.state.global_step}")
print(f"  Runtime: {sft_result.metrics['train_runtime']:.1f} seconds")

Applying formatting function to train dataset:   0%|          | 0/408 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/408 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/408 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/102 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/102 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/102 [00:00<?, ? examples/s]

STARTING LoRA INSTRUCTION FINE-TUNING
  Model:          ./outputs/cpt_model/
  Adapter:        B (Balanced) — r=16, α=32
  Max steps:      10 (-1 = all)
  Batch size:     1 (effective: 1)
  Learning rate:  0.0002
  Max seq length: 512
  ⚠️  CPU mode: limited to 10 steps


Step,Training Loss
2,2.349616
4,3.340255
6,3.010938
8,2.920611
10,3.198633



LoRA SFT TRAINING COMPLETE
  Final loss: 2.9640
  Total steps: 10
  Runtime: 318.6 seconds


In [29]:
# ============================================================
# Cell B2.5 — Save QLoRA Adapter
# ============================================================

sft_model.save_pretrained(SFT_CHECKPOINT_DIR)
tokenizer.save_pretrained(SFT_CHECKPOINT_DIR)

print(f"QLoRA adapter saved to: {SFT_CHECKPOINT_DIR}")
print(f"Adapter size: {sum(f.stat().st_size for f in Path(SFT_CHECKPOINT_DIR).rglob('*') if f.is_file()) / 1e6:.1f} MB")

QLoRA adapter saved to: ./outputs/sft_model/
Adapter size: 7.8 MB


### Inference — Part B2

- **QLoRA** combines 4-bit quantization of the base model with low-rank adapters, enabling fine-tuning of large models on consumer GPUs.
- **Adapter B (Balanced)** with r=16, \u03b1=32 provides a good trade-off between model quality and training cost.
- **NF4 quantization** (NormalFloat4) is information-theoretically optimal for normally-distributed weights.
- **Paged AdamW 8-bit** optimizer reduces memory footprint by ~50% compared to standard AdamW.
- The adapter weights are typically only 10-50 MB, making deployment lightweight.

---
## Part B3 — Evaluation & Comparative Analysis [1 Mark]

Run the fine-tuned adapter on the same 3 domain prompts and compare with baseline.

In [30]:
# ============================================================
# Cell B3.1 — Load Fine-Tuned Model for Inference
# ============================================================

if torch.cuda.is_available():
    eval_base_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    eval_base_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float32,
    )

# Load the LoRA adapter on top
eval_model = PeftModel.from_pretrained(eval_base_model, SFT_CHECKPOINT_DIR)
eval_model.eval()

print(f"Fine-tuned model loaded with adapter from: {SFT_CHECKPOINT_DIR}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie biogpt.embed_tokens.weight to output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Fine-tuned model loaded with adapter from: ./outputs/sft_model/


In [31]:
# ============================================================
# Cell B3.2 — Comparative Evaluation: Baseline vs Fine-Tuned
# ============================================================
from IPython.display import display, HTML

def generate_completion(model, tokenizer, prompt: str, max_new_tokens: int = 200) -> str:
    """Generate text continuation from a partial-sentence prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.92,
            temperature=0.7,
            repetition_penalty=1.3,
            num_beams=1,
        )
    
    generated = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return generated.strip()


# --- Load original base model (pre-CPT) for baseline ---
print("Loading original base model for baseline comparison...")
base_model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32
)
base_model_eval.eval()

# Completion-style prompts (partial sentences BioGPT can naturally continue)
DOMAIN_PROMPTS_COMPLETION = [
    "The first-line treatment for uncomplicated community-acquired pneumonia in adults is",
    "The recommended dosing guidelines for Vancomycin in patients with renal impairment include",
    "The WHO criteria for initiating antiretroviral therapy in adults with HIV state that",
]

print(f"\nGenerating responses from both models...")
comparison_results = []
for i, (question, completion_prompt) in enumerate(zip(DOMAIN_PROMPTS, DOMAIN_PROMPTS_COMPLETION), 1):
    # --- Baseline (original pre-trained BioGPT) ---
    base_generated = generate_completion(base_model_eval, tokenizer, completion_prompt, max_new_tokens=150)
    base_output = completion_prompt + " " + base_generated
    
    # --- Fine-Tuned (CPT + QLoRA SFT model) ---
    ft_generated = generate_completion(eval_model, tokenizer, completion_prompt, max_new_tokens=150)
    ft_output = completion_prompt + " " + ft_generated
    
    comparison_results.append({
        "Prompt #": i,
        "Question": question,
        "Baseline Response\n(Original BioGPT)": base_output,
        "Fine-Tuned Response\n(CPT + QLoRA SFT)": ft_output,
    })

# Free base model
del base_model_eval
import gc; gc.collect()

# --- Display as formatted HTML table ---
comp_df = pd.DataFrame(comparison_results)

html = """
<h3 style="margin-bottom:10px;">Comparative Evaluation: Baseline vs Fine-Tuned (CPT + QLoRA)</h3>
<p style="font-size:12px; color:#555;">Both models use the same completion-style prompt. The fine-tuned model should show improved clinical accuracy.</p>
<table border="1" cellpadding="10" cellspacing="0" style="border-collapse:collapse; width:100%; font-size:13px; line-height:1.5;">
<thead>
<tr style="background-color:#2c3e50; color:white;">
    <th style="width:4%;">#</th>
    <th style="width:18%;">Clinical Question</th>
    <th style="width:39%;">Baseline Response<br>(Original BioGPT)</th>
    <th style="width:39%;">Fine-Tuned Response<br>(CPT + QLoRA SFT)</th>
</tr>
</thead>
<tbody>
"""

for _, row in comp_df.iterrows():
    baseline_text = row['Baseline Response\n(Original BioGPT)'].replace('\n', '<br>')
    ft_text = row['Fine-Tuned Response\n(CPT + QLoRA SFT)'].replace('\n', '<br>')
    html += f"""<tr style="vertical-align:top;">
    <td style="text-align:center; font-weight:bold; background-color:#ecf0f1;">{row['Prompt #']}</td>
    <td style="font-weight:bold; background-color:#ecf0f1;">{row['Question']}</td>
    <td style="white-space:pre-wrap; padding:10px;">{baseline_text[:600]}</td>
    <td style="white-space:pre-wrap; padding:10px; background-color:#eafaf1;">{ft_text[:600]}</td>
</tr>"""

html += "</tbody></table>"
html += f'<p style="font-size:11px; color:#888; margin-top:8px;"><em>{DISCLAIMER}</em></p>'

display(HTML(html))

# Also print plain-text version
print(f"\n{'='*80}")
print("PLAIN TEXT SUMMARY")
print(f"{'='*80}")
separator = '\u2500' * 80
for _, row in comp_df.iterrows():
    print(f"\n{separator}")
    print(f"  Question {row['Prompt #']}: {row['Question']}")
    print(separator)
    baseline_key = 'Baseline Response\n(Original BioGPT)'
    ft_key = 'Fine-Tuned Response\n(CPT + QLoRA SFT)'
    print(f"\n  [BASELINE - Original BioGPT]")
    print(f"  {row[baseline_key][:500]}")
    print(f"\n  [FINE-TUNED - CPT + QLoRA SFT]")

    print(f"  {row[ft_key][:500]}")
print(f"\n{'='*80}")
print(f"\n{DISCLAIMER}")

Loading original base model for baseline comparison...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie biogpt.embed_tokens.weight to output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Generating responses from both models...


#,Clinical Question,Baseline Response(Original BioGPT),Fine-Tuned Response(CPT + QLoRA SFT)
1,What is the first-line treatment for uncomplicated community-acquired pneumonia in adults?,The first-line treatment for uncomplicated community-acquired pneumonia in adults is a beta lactam and an aminoglycoside.,"The first-line treatment for uncomplicated community-acquired pneumonia in adults is a 7 to 14 day course of antibiotics, preferably beta-lactam monotherapy."
2,What are the recommended dosing guidelines for Vancomycin in patients with renal impairment?,"The recommended dosing guidelines for Vancomycin in patients with renal impairment include a dose adjustment based on creatinine clearance (CL / F), but this has not been validated.","The recommended dosing guidelines for Vancomycin in patients with renal impairment include a target trough concentration of 15-20 mg / L, which is an established dose to maintain therapeutic drug levels."
3,What are the WHO criteria for initiating antiretroviral therapy in adults with HIV?,The WHO criteria for initiating antiretroviral therapy in adults with HIV state that all patients have an undetectable viral load.,"The WHO criteria for initiating antiretroviral therapy in adults with HIV state that the CD4 cell count should be < 350 cells / mm3, and the World Health Organization (WHO) guidelines provide a range of cut-offs from 100 to 500 cells / mm3."



PLAIN TEXT SUMMARY

────────────────────────────────────────────────────────────────────────────────
  Question 1: What is the first-line treatment for uncomplicated community-acquired pneumonia in adults?
────────────────────────────────────────────────────────────────────────────────

  [BASELINE - Original BioGPT]
  The first-line treatment for uncomplicated community-acquired pneumonia in adults is a beta lactam and an aminoglycoside.

  [FINE-TUNED - CPT + QLoRA SFT]
  The first-line treatment for uncomplicated community-acquired pneumonia in adults is a 7 to 14 day course of antibiotics, preferably beta-lactam monotherapy.

────────────────────────────────────────────────────────────────────────────────
  Question 2: What are the recommended dosing guidelines for Vancomycin in patients with renal impairment?
────────────────────────────────────────────────────────────────────────────────

  [BASELINE - Original BioGPT]
  The recommended dosing guidelines for Vancomycin in patien

### Inference — Part B3: Observations

**Expected improvements after QLoRA fine-tuning:**

1. **Correctness:** The fine-tuned model should produce more accurate clinical protocol information, closely matching the source guidelines (WHO, CDC, NICE, ICMR).

2. **Domain Terminology:** Outputs should use proper clinical vocabulary (e.g., "first-line treatment", "CrCl", "TDM monitoring", "ART initiation criteria") rather than generic language.

3. **Completeness:** Responses should be more structured and comprehensive \u2014 covering dosage, duration, contraindications, and monitoring requirements.

4. **Hallucination Risk:** Monitor for hallucinated drug dosages or protocols not present in the training corpus. The instruction-grounded approach reduces but does not eliminate this risk.

5. **Disclaimer Compliance:** Fine-tuned responses should consistently include the clinical disclaimer as trained.

**Note:** *This output is for educational/reference purposes only and must not replace professional clinical judgment.*

---
## Optional Extension — Interactive Clinical Protocol Lookup

A simple CLI/notebook chat loop for querying the clinical protocol assistant.

In [32]:
# ============================================================
# Cell EXT.1 — Interactive Clinical Protocol Chat Loop
# ============================================================

def generate_instruction_response(model, tokenizer, instruction: str, max_new_tokens: int = 250) -> str:
    """Generate a response to an instruction using the fine-tuned model."""
    prompt = f"""### Instruction:
{instruction}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.92,
            temperature=0.7,
            repetition_penalty=1.3,
            num_beams=1,
        )
    
    generated = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return generated.strip()


def clinical_chat(model, tokenizer, max_new_tokens: int = 250):
    """Interactive chat loop for clinical protocol queries."""
    print("="*60)
    print("CLINICAL PROTOCOL LOOKUP ASSISTANT")
    print("="*60)
    print("Ask clinical protocol questions. Type 'quit' to exit.")
    print(f"Disclaimer: {DISCLAIMER}")
    print("="*60)
    
    while True:
        user_input = input("\n\U0001fa7a Your question: ").strip()
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Exiting clinical assistant. Goodbye!")
            break
        
        if not user_input:
            continue
        
        response = generate_instruction_response(model, tokenizer, user_input, max_new_tokens)
        
        if DISCLAIMER not in response:
            response += f"\n\n{DISCLAIMER}"
        
        print(f"\n\U0001f48a Response:\n{response}")


# Uncomment to run the interactive chat:
# clinical_chat(eval_model, tokenizer)

---
## Summary

| Metric | Value |
|--------|-------|
| Model | BioGPT (347M) |
| Domain | Medical & Clinical Literature |
| Variant | V4 \u2014 Clinical Protocol Lookup |
| LoRA Adapter | Adapter B (r=16, \u03b1=32) |
| Target Modules | q_proj, v_proj |
| SFT Epochs | 3 (GPU) / 10 steps (CPU) |
| Max Seq Length | 512 |

---
*This notebook is for educational/reference purposes only and must not replace professional clinical judgment.*